# Segmento 1 — Configurazione ed esplorazione del dataset

Scaricheremo un dataset di ~13.500 ricette da Hugging Face ed esploreremo come si presentano i dati.

In [1]:
from datasets import load_dataset
import pandas as pd

/home/federios/aidea/boolean-master/boolean-master-demo/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Scaricare il dataset

Il dataset è ospitato su Hugging Face: [Hieu-Pham/kaggle_food_recipes](https://huggingface.co/datasets/Hieu-Pham/kaggle_food_recipes). Contiene ~13.500 ricette estratte da Epicurious, con titoli, ingredienti e istruzioni. Rimuoviamo le colonne relative alle immagini poiché non le utilizzeremo.

In [2]:
ds = load_dataset("Hieu-Pham/kaggle_food_recipes", split="train")
df = ds.to_pandas().drop(columns=["Unnamed: 0", "Image_Name"])
print(f"{len(df)} recipes, columns: {list(df.columns)}")
df.head()

13501 recipes, columns: ['Title', 'Ingredients', 'Instructions', 'Cleaned_Ingredients']


,Title,Ingredients,Instructions,Cleaned_Ingredients
0,Miso-Butter Roast Chicken With Acorn Squash Pa...,"['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher...","Pat chicken dry with paper towels, season all ...","['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher..."
1,Crispy Salt and Pepper Potatoes,"['2 large egg whites', '1 pound new potatoes (...",Preheat oven to 400°F and line a rimmed baking...,"['2 large egg whites', '1 pound new potatoes (..."
2,Thanksgiving Mac and Cheese,"['1 cup evaporated milk', '1 cup whole milk', ...",Place a rack in middle of oven; preheat to 400...,"['1 cup evaporated milk', '1 cup whole milk', ..."
3,Italian Sausage and Bread Stuffing,"['1 (¾- to 1-pound) round Italian loaf, cut in...",Preheat oven to 350°F with rack in middle. Gen...,"['1 (¾- to 1-pound) round Italian loaf, cut in..."
4,Newton's Law,"['1 teaspoon dark brown sugar', '1 teaspoon ho...",Stir together brown sugar and hot water in a c...,"['1 teaspoon dark brown sugar', '1 teaspoon ho..."


## Com'è fatta una ricetta?

Guardiamo una singola ricetta nel dettaglio per capire i dati grezzi.

In [3]:
recipe = df.iloc[0]

print(f"Title: {recipe['Title']}")
print(f"\n--- Ingredients ---\n{recipe['Ingredients']}")
print(f"\n--- Instructions ---\n{recipe['Instructions']}")

Title: Miso-Butter Roast Chicken With Acorn Squash Panzanella

--- Ingredients ---
['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher salt, divided, plus more', '2 small acorn squash (about 3 lb. total)', '2 Tbsp. finely chopped sage', '1 Tbsp. finely chopped rosemary', '6 Tbsp. unsalted butter, melted, plus 3 Tbsp. room temperature', '¼ tsp. ground allspice', 'Pinch of crushed red pepper flakes', 'Freshly ground black pepper', '⅓ loaf good-quality sturdy white bread, torn into 1" pieces (about 2½ cups)', '2 medium apples (such as Gala or Pink Lady; about 14 oz. total), cored, cut into 1" pieces', '2 Tbsp. extra-virgin olive oil', '½ small red onion, thinly sliced', '3 Tbsp. apple cider vinegar', '1 Tbsp. white miso', '¼ cup all-purpose flour', '2 Tbsp. unsalted butter, room temperature', '¼ cup dry white wine', '2 cups unsalted chicken broth', '2 tsp. white miso', 'Kosher salt, freshly ground pepper']

--- Instructions ---
Pat chicken dry with paper towels, season all over with 2 tsp. sal

Nota che `Ingredients` è una lista Python in formato stringa — non testo semplice. `Cleaned_Ingredients` è simile ma separa gli elementi combinati (es. "salt, pepper") in voci distinte. Facciamo il parsing e confrontiamoli, poi guardiamo qualche altra ricetta per familiarizzare con i dati.

In [4]:
import ast

# Parse the stringified lists into actual Python lists
recipe = df.iloc[0]
ingredients_raw = ast.literal_eval(recipe["Ingredients"])
ingredients_cleaned = ast.literal_eval(recipe["Cleaned_Ingredients"])

print("Raw ingredients (first 5):")
for ing in ingredients_raw[:5]:
    print(f"  - {ing}")

print("\nCleaned ingredients (first 5):")
for ing in ingredients_cleaned[:5]:
    print(f"  - {ing}")

Raw ingredients (first 5):
  - 1 (3½–4-lb.) whole chicken
  - 2¾ tsp. kosher salt, divided, plus more
  - 2 small acorn squash (about 3 lb. total)
  - 2 Tbsp. finely chopped sage
  - 1 Tbsp. finely chopped rosemary

Cleaned ingredients (first 5):
  - 1 (3½–4-lb.) whole chicken
  - 2¾ tsp. kosher salt, divided, plus more
  - 2 small acorn squash (about 3 lb. total)
  - 2 Tbsp. finely chopped sage
  - 1 Tbsp. finely chopped rosemary


In [5]:
import textwrap

# Look at a few more recipes
for i in [1, 5, 42]:
    r = df.iloc[i]
    print(f"\n{'='*60}")
    print(f"{r['Title']}")
    print(f"{'='*60}")
    ingredients = ast.literal_eval(r["Ingredients"])
    print(f"\nIngredients ({len(ingredients)}):")
    for ing in ingredients[:6]:
        print(f"  - {ing}")
    if len(ingredients) > 6:
        print(f"  ... and {len(ingredients) - 6} more")
    print(f"\nInstructions:")
    print(textwrap.fill(r["Instructions"][:1000], width=80))
    if len(r["Instructions"]) > 1000:
        print("...")


Crispy Salt and Pepper Potatoes

Ingredients (7):
  - 2 large egg whites
  - 1 pound new potatoes (about 1 inch in diameter)
  - 2 teaspoons kosher salt
  - ¾ teaspoon finely ground black pepper
  - 1 teaspoon finely chopped rosemary
  - 1 teaspoon finely chopped thyme
  ... and 1 more

Instructions:
Preheat oven to 400°F and line a rimmed baking sheet with parchment. In a large
bowl, whisk the egg whites until foamy (there shouldn’t be any liquid whites in
the bowl). Add the potatoes and toss until they’re well coated with the egg
whites, then transfer to a strainer or colander and let the excess whites drain.
Season the potatoes with the salt, pepper, and herbs. Scatter the potatoes on
the baking sheet (make sure they’re not touching) and roast until the potatoes
are very crispy and tender when poked with a knife, 15 to 20 minutes (depending
on the size of the potatoes). Transfer to a bowl and serve.

Warm Comfort

Ingredients (4):
  - 2 chamomile tea bags
  - 1½ oz. reposado tequil

## Panoramica rapida del dataset

In [6]:
# How many ingredients per recipe?
df["n_ingredients"] = df["Ingredients"].apply(lambda x: len(ast.literal_eval(x)))

print(f"Total recipes: {len(df)}")
print(f"\nIngredients per recipe:")
print(df["n_ingredients"].describe().to_string())
print(f"\nInstruction length (characters):")
print(df["Instructions"].str.len().describe().to_string())

Total recipes: 13501

Ingredients per recipe:
count    13501.000000
mean        10.643138
std          4.830709
min          0.000000
25%          7.000000
50%         10.000000
75%         13.000000
max         51.000000

Instruction length (characters):
count    13495.000000
mean      1040.520119
std        711.006395
min          1.000000
25%        569.000000
50%        890.000000
75%       1344.500000
max      13952.000000


In [7]:
# A few more sample recipes — short and long
short = df.nsmallest(1, "n_ingredients").iloc[0]
long = df.nlargest(1, "n_ingredients").iloc[0]

for label, r in [("Simplest", short), ("Most complex", long)]:
    ingredients = ast.literal_eval(r["Ingredients"])
    print(f"\n{'='*60}")
    print(f"{label}: {r['Title']} ({len(ingredients)} ingredients)")
    print(f"{'='*60}")
    print(f"\nIngredients: {ingredients[:8]}{'...' if len(ingredients) > 8 else ''}")
    print(f"\nInstructions:")
    print(textwrap.fill(r["Instructions"], width=80))


Simplest: Sautéed Shishito Peppers: Summer's Best New Bite (0 ingredients)

Ingredients: []

Instructions:
Here's what you do. Heat a little olive oil in a wide sauté pan until it is good
and hot but not smoking. Add the peppers and cook them over medium, tossing and
turning them frequently until they blister. They shouldn't char except in
places. Don't rush. It takes 10 to 15 minutes to cook a panful of peppers. When
they're done, toss them with sea salt and add a squeeze of fresh lemon. Slide
the peppers into a bowl and serve them hot. You pick them up by the stem end and
eat the whole thing, minus the stem, that is. You can probably do fancier,
cheffy things with them, but they're terrific like this. For variety, I
sometimes use a little toasted sesame oil instead of olive oil and finish them
with togarashi. If you have leftovers, an unlikely event in my experience, chop
off the stems and put the peppers in an omelet or some scrambled eggs.

Most complex: Epi's 50-Ingredient Super 